# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the TensorDict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` holds preset constructors (`cartpole`, `procedural_frozenlake`, `synthetic`, `atari`, …). `make_vector_env` wraps Gymnasium and formats steps as `(data, metadata, metrics)`.

In [ ]:
from mouse.envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=4` runs four CartPole instances in parallel.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [ ]:
cfg = EnvConfig.cartpole(seed=0, num_envs=4, max_episode_steps=500)
env = make_vector_env(cfg)

## Rollout loop

Each `env.step` returns:

- **`data[i]`** — `time`, `observation` (dict: `discrete` / `continuous` / `image`), `reward`, `done`, …
- **`metadata`** — shared dict with per-env arrays: `metadata["group_ids"][i]`, `metadata["episode_index"][i]`, `metadata["reward_episodic"][i]`, optional `metadata["q_star"][i]`
- **`metrics[i]`** — `episode_cum_reward` and `episode_length` lists for finishes on this step (empty if none)

Each **`actions[i]["action"]`** passed to `step()` is also a dict — `discrete` or `continuous`, matching the env's action space.

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [ ]:
for step in range(1000):
    actions = env.sample_random_actions()
    data, metadata, metrics = env.step(actions)

    if step % 100 == 0:
        group_ids = metadata["group_ids"]
        reward_episodic = metadata["reward_episodic"]
        for i, td in enumerate(data):
            print(
                f"step={step:4d}  "
                f"group_id={group_ids[i]}  "
                f"time={td['time'].item()}  "
                f"done={td['done'].item()}  "
                f"reward={td['reward'].item():.3f}  "
                f"reward_episodic={reward_episodic[i]:.3f}"
            )

## Cleanup

In [ ]:
env.close()